# 02 · Auto Loader — ingesta incremental de obras CFE

**Auto Loader** (`cloudFiles`) ingiere archivos nuevos de forma **incremental y con estado**:
recuerda qué archivos ya procesó, así que en cada corrida solo lee lo nuevo.

En este lab verás:
1. Preparar una **landing zone** con lotes de datos de obras.
2. Ingesta incremental con `trigger(availableNow=True)` — primer lote.
3. **Simular llegada de datos nuevos** — segundo lote (Auto Loader solo lee lo nuevo).
4. `cloud_files_state()` — inspeccionar qué archivos ha descubierto el checkpoint.
5. Columnas de **linaje** (`_metadata`) para trazabilidad.
6. **Schema evolution** — qué pasa cuando el origen agrega campos nuevos.

> Usamos la carpeta `ejercicio1/` que contiene un solo archivo `tbobras.parquet`
> (318 obras de construcción). Para simular la llegada de datos nuevos, generaremos
> un segundo archivo directamente en esa carpeta.

In [0]:
%run ./_setup

## 1. Primera ingesta incremental
Leemos la carpeta `ejercicio1/` que ya tiene `tbobras.parquet` (318 obras).
`availableNow=True` procesa todo lo disponible y **se detiene** (ideal para batch incremental).

In [0]:
from pyspark.sql import functions as F

# --- rutas ---
bronze     = tbl("bronze_obras_cfe")
chk        = f"{CHK_PATH}/bronze_obras_cfe"
schema_loc = f"{CHK_PATH}/bronze_obras_cfe_schema"

# --- limpieza (para re-ejecutar desde cero) ---
spark.sql(f"DROP TABLE IF EXISTS {bronze}")
for p in (chk, schema_loc):
    dbutils.fs.rm(p, True)

# --- Auto Loader: primera ingesta desde ejercicio1/ ---
q = (spark.readStream
   .format("cloudFiles")
   .option("cloudFiles.format", "parquet")
   .option("cloudFiles.schemaLocation", schema_loc)
   .option("rescuedDataColumn", "_rescued_data")
   .load(EJERCICIO1_PATH)
 .writeStream
   .option("checkpointLocation", chk)
   .trigger(availableNow=True)
   .toTable(bronze))
q.awaitTermination()

print(f"Carpeta fuente: {EJERCICIO1_PATH}")
print(f"✅ Bronce: {spark.table(bronze).count():,} registros")

## 2. Llegan datos nuevos
Simulamos la llegada de un **segundo archivo** con 10 obras actualizadas.
Al re-ejecutar Auto Loader, solo procesa el archivo nuevo — el checkpoint ya
registró `tbobras.parquet`.

In [0]:
before = spark.table(bronze).count()

# Generar un segundo archivo con 10 obras "actualizadas"
df_nuevas = (spark.read.parquet(DELTA_LAB_FILE)
    .limit(10)
    .withColumn("SituacionActual", F.lit("Actualizado en Workshop")))
df_nuevas.write.mode("overwrite").parquet(f"{EJERCICIO1_PATH}/lote_2_actualizacion.parquet")
print(f"Lote 2 depositado: {df_nuevas.count()} obras en {EJERCICIO1_PATH}")

# Re-ejecutar Auto Loader — solo procesa el archivo nuevo
q = (spark.readStream.format("cloudFiles")
   .option("cloudFiles.format", "parquet")
   .option("cloudFiles.schemaLocation", schema_loc)
   .option("rescuedDataColumn", "_rescued_data")
   .load(EJERCICIO1_PATH)
 .writeStream.option("checkpointLocation", chk)
   .trigger(availableNow=True).toTable(bronze))
q.awaitTermination()

after = spark.table(bronze).count()
print(f"\nAntes: {before:,}  |  Después: {after:,}  |  Nuevas: {after - before:,}")
print("✅ Solo se procesaron las obras del lote 2 — tbobras.parquet NO se reproc.")

In [0]:
df_state = spark.sql(f"""
    SELECT path, size, create_time, discovery_time, processed_time
    FROM cloud_files_state('{chk}')
    ORDER BY discovery_time DESC
""")
print(f"Archivos registrados en el checkpoint: {df_state.count()}")
display(df_state)

### 2c. Columnas de linaje con `_metadata`

| Campo | Tipo | Uso |
|---|---|---|
| `_metadata.file_path` | STRING | Trazabilidad: de qué archivo vino |
| `_metadata.file_name` | STRING | Identificar archivo sin ruta completa |
| `_metadata.file_size` | LONG | Detectar archivos vacíos o anómalos |
| `_metadata.file_modification_time` | TIMESTAMP | Detectar re-escrituras |

In [0]:
# Leer desde la landing zone para ver de qué archivo vino cada registro
df_with_meta = (
    spark.read.format("parquet").load(EJERCICIO1_PATH)
    .select(
        "IdObra", "NombreObra",
        F.col("_metadata.file_name").alias("source_file"),
        F.col("_metadata.file_size").alias("bytes"),
        F.col("_metadata.file_modification_time").alias("modified_at"),
    )
)

# Resumen por archivo fuente
display(
    df_with_meta
    .groupBy("source_file", "bytes")
    .agg(F.count("*").alias("registros"))
    .orderBy("source_file")
)

## 5. Schema evolution
| Modo | ¿Falla? | Comportamiento |
|---|---|---|
| **`addNewColumns`** | Sí (1 vez) | Agrega columna nueva al reiniciar |
| **`rescue`** | No | Columnas nuevas van a `_rescued_data` |
| **`failOnNewColumns`** | Sí | Requiere intervención manual |
| **`none`** | No | Ignora columnas nuevas |

Simulemos la llegada de un archivo Parquet con columnas adicionales:

In [0]:
# Preparar sandbox APARTE para schema evolution
landing_evol = f"{CHK_PATH}/landing_evol"
chk_evol     = f"{CHK_PATH}/evol_stream"
sloc_evol    = f"{CHK_PATH}/evol_schema"
for p in (landing_evol, chk_evol, sloc_evol): dbutils.fs.rm(p, True)
dbutils.fs.mkdirs(landing_evol)

evol_bronze = tbl("bronze_obras_evol")
spark.sql(f"DROP TABLE IF EXISTS {evol_bronze}")

# Copiar tbobras.parquet como lote base
dbutils.fs.cp(DELTA_LAB_FILE, f"{landing_evol}/lote_1_obras.parquet")
print("Lote 1 copiado: tbobras.parquet (esquema original)")

def run_evol_autoloader():
    q = (spark.readStream.format("cloudFiles")
       .option("cloudFiles.format", "parquet")
       .option("cloudFiles.schemaLocation", sloc_evol)
       .option("cloudFiles.schemaEvolutionMode", "rescue")
       .option("rescuedDataColumn", "_rescued_data")
       .load(landing_evol)
     .writeStream.option("checkpointLocation", chk_evol)
       .trigger(availableNow=True).toTable(evol_bronze))
    q.awaitTermination()

run_evol_autoloader()
print(f"Tras lote BASE → {spark.table(evol_bronze).count():,} filas")

In [0]:
# Crear Parquet con columnas NUEVAS (simulando evolución de esquema)
df_evol = (spark.read.parquet(f"{EJERCICIO1_PATH}/tbobras.parquet")
    .limit(10)
    .withColumn("prioridad_obra", F.lit("Alta"))
    .withColumn("costo_estimado_usd", F.col("ImporteContratoMxn") / 17.5))

df_evol.write.mode("overwrite").parquet(f"{landing_evol}/lote_2_evolucion.parquet")
print("Lote 2 depositado con columnas nuevas: prioridad_obra, costo_estimado_usd")

run_evol_autoloader()

count = spark.table(evol_bronze).count()
rescued = spark.table(evol_bronze).where("_rescued_data IS NOT NULL").count()
print(f"Total filas: {count:,}  |  Filas con _rescued_data: {rescued}")

print("\n🔍 Filas con datos 'rescatados' (columnas nuevas no reconocidas por el esquema original):")
display(spark.table(evol_bronze)
   .where("_rescued_data IS NOT NULL")
   .select("IdObra", "NombreObra", "_rescued_data")
   .limit(5))

In [0]:
# Eliminar el archivo que generamos en ejercicio1 para dejar la carpeta limpia
try:
    dbutils.fs.rm(f"{EJERCICIO1_PATH}/lote_2_actualizacion.parquet", True)
    print("🧹 lote_2_actualizacion.parquet eliminado de ejercicio1/")
except:
    print("No había archivo que limpiar")

# Verificar que solo queda el archivo original
for f in dbutils.fs.ls(EJERCICIO1_PATH):
    print(f"  {f.name}  ({f.size:,} bytes)")